# Start

This project focuses on a clustering exercise on the trips made with the public bicycles of London (Santander Cycles) in 2025. The idea is to work with the raw data, clean it and transform it into something useful for the clustering exercise. We want to analyse and cluster the stations using 2 different representations: a **vector representation of the stations** and a **graph representation of the flows between stations**. The workflow is the following:

1. Load all the data (multiple .csv files)
2. Clean raw data
3. Analyze the data and transform it
4. Clustering exercise 1: Station analysis without considering the network
    - Transform data to create station profiles
    - Dimension reduction techniques
    - Clustering of data
    - Anomaly detection
    - Model conclusion
5. Clustering exercise 2: Station analysis considering the network
    - Transform data to create origin-destination features
    - Stochastic block model (SBM)
    - Model conclusion
6. Model comparison
7. Final conclusion

All these steps are divided in 2 notebooks. The first one tackles the first 2 points and the second one tackles the other 4.



# 1. Load all data
After downloading all files, they will all be loaded using Polars and will be merged into one single DF

In [ ]:
import polars as pl
from pathlib import Path

In [2]:
files_path = Path("../data/input_raw").glob("*.csv")

for file in Path("../data/input_raw").glob("*.csv"):
    df = pl.read_csv(file)

    print(file)
    print(df.columns)
    print("----------------")

# the files have the same column names but not in the same order


../data/input_raw/427JourneyDataExtract01Sep2025-15Sep2025.csv
['Number', 'Start date', 'Start station', 'Start station number', 'End date', 'End station', 'End station number', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------
../data/input_raw/420JourneyDataExtract14May2025-31May2025.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------
../data/input_raw/414JourneyDataExtract15Feb2025-28Feb2025.csv
['Number', 'Start date', 'Start station number', 'Start station', 'End date', 'End station number', 'End station', 'Bike number', 'Bike model', 'Total duration', 'Total duration (ms)']
----------------
../data/input_raw/433JourneyDataExtract01Dec2025-15Dec2025.csv
['Number', 'Start date', 'Start station number', 'End date', 'End station', 'End station number', 'Start station', 'Bike number', 'Bike model'

In [3]:
################
# Read the files
################
files = sorted(Path("../data/input_raw").glob("*.csv"))

# Use first file order
first_df = pl.read_csv(files[0])
columns = first_df.columns

#Read and reorder files
dfs = [pl.read_csv(file).select(columns)
       for file in files]

df = pl.concat(dfs)

print(df.head())
print(df.shape)
print("----------------------")
# All dfs are grouped into one giant df of shape (9068241, 11)

shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Number    ┆ Start     ┆ Start     ┆ Start     ┆ … ┆ Bike      ┆ Bike      ┆ Total     ┆ Total    │
│ ---       ┆ date      ┆ station   ┆ station   ┆   ┆ number    ┆ model     ┆ duration  ┆ duration │
│ i64       ┆ ---       ┆ number    ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ (ms)     │
│           ┆ str       ┆ ---       ┆ str       ┆   ┆ i64       ┆ str       ┆ str       ┆ ---      │
│           ┆           ┆ i64       ┆           ┆   ┆           ┆           ┆           ┆ i64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 145666816 ┆ 2025-01-1 ┆ 1043      ┆ Museum of ┆ … ┆ 55223     ┆ CLASSIC   ┆ 14m 30s   ┆ 870468   │
│           ┆ 4 23:59   ┆           ┆ London,   ┆   ┆           ┆           ┆           ┆          │
│           ┆           ┆           ┆ Barbican  ┆   ┆           ┆           

# 2. Clean raw data
We will then do some basic data cleaning of the whole dataset to make sure that we keep only the necessary information + some basic analysis on it

In [ ]:
#######################
#Filter df to keep only useful columns
#######################


columns_to_drop = ["Number", "Start station", "End station", "Bike number", "Bike model", "Total duration"]

df = df.drop(columns_to_drop)

print(df.head())
print(df.columns)
print(df.shape)

shape: (5, 5)
┌──────────────────┬───────────────┬──────────────────┬──────────────────────┬─────────────────────┐
│ Start date       ┆ Start station ┆ End date         ┆ End station number   ┆ Total duration (ms) │
│ ---              ┆ number        ┆ ---              ┆ ---                  ┆ ---                 │
│ str              ┆ ---           ┆ str              ┆ i64                  ┆ i64                 │
│                  ┆ i64           ┆                  ┆                      ┆                     │
╞══════════════════╪═══════════════╪══════════════════╪══════════════════════╪═════════════════════╡
│ 2025-01-14 23:59 ┆ 1043          ┆ 2025-01-15 00:13 ┆ 200149               ┆ 870468              │
│ 2025-01-14 23:59 ┆ 300015        ┆ 2025-01-15 00:04 ┆ 300229               ┆ 307181              │
│ 2025-01-14 23:59 ┆ 1068          ┆ 2025-01-15 00:10 ┆ 1051                 ┆ 640755              │
│ 2025-01-14 23:59 ┆ 1159          ┆ 2025-01-15 00:10 ┆ 1007                 

In [5]:
###################
#Export csv
###################

#df.write_csv("../data/output/v2/1.full_filtered_df.csv")
df.write_parquet("../data/output/v2/1.full_filtered_df.parquet")

One of the basic things to consider is the existence of missing values. There are 184 missing values in the columns 'End station number' and 'Total duration (ms)'. After `drop_nulls()` the dataframe goes from 9,068,241 to 9,068,057 rows, i.e. exactly 184 rows are removed: **the missing values are on the same rows** (trips without an end station also have no duration: most likely bikes that were never docked). Removing them is a loss of 0.002% of the data.

In [6]:
########################
#Missing start or end station number
########################
missing_values = df.null_count()

print(missing_values)

#184 missing values for columns end station number and total duration. if the missing values for the entries are not the same we have a total of 
# 184*2 = 368 out of 9068241 entries. Removing them would be a loss of less than 1% of the dataframe

df_no_nulls = df.drop_nulls()
print(df_no_nulls.shape)
print("---------------")

shape: (1, 5)
┌────────────┬──────────────────────┬──────────┬────────────────────┬─────────────────────┐
│ Start date ┆ Start station number ┆ End date ┆ End station number ┆ Total duration (ms) │
│ ---        ┆ ---                  ┆ ---      ┆ ---                ┆ ---                 │
│ u32        ┆ u32                  ┆ u32      ┆ u32                ┆ u32                 │
╞════════════╪══════════════════════╪══════════╪════════════════════╪═════════════════════╡
│ 0          ┆ 0                    ┆ 0        ┆ 184                ┆ 184                 │
└────────────┴──────────────────────┴──────────┴────────────────────┴─────────────────────┘
(9068057, 5)
---------------


Another point to consider is the "impossible" trips. This means trips that last for 0 seconds or that last for less than that. After removing the missing values of the DF, we can see that there are no negative or 0 second entries on the DF.

In [7]:
########################
#Total duration = 0, missing or negative
########################

df_duration = df_no_nulls.filter(pl.col("Total duration (ms)") <= 0)
print(df_duration)
print("-------------------")

# After removing the missing entries for the total duration, we see there are no negative or 0 ms entries 


shape: (0, 5)
┌────────────┬──────────────────────┬──────────┬────────────────────┬─────────────────────┐
│ Start date ┆ Start station number ┆ End date ┆ End station number ┆ Total duration (ms) │
│ ---        ┆ ---                  ┆ ---      ┆ ---                ┆ ---                 │
│ str        ┆ i64                  ┆ str      ┆ i64                ┆ i64                 │
╞════════════╪══════════════════════╪══════════╪════════════════════╪═════════════════════╡
└────────────┴──────────────────────┴──────────┴────────────────────┴─────────────────────┘
-------------------


We can also check round trips. A round trip is defined as a trip whose start station is the same as its end station. These trips carry no origin-destination information (they would be self-loops in the flow graph of notebook 2) and we cannot know where the user actually went, so we identify them and remove them from the data.

In [8]:
########################
#same start and end station number
########################

#are there entries where the start and end stations are the same?
df_same_station = df_no_nulls.filter(pl.col("Start station number") == pl.col("End station number"))
print(df_same_station)

#305702 entries start and end in the same station. this represents 305702/9068057 = 0.033 -> 3.3% of the no_nulls dataset

#Lets filter them out
df_no_same_station = df_no_nulls.filter(pl.col("Start station number") != pl.col("End station number"))
print(df_no_same_station)

#new df shape is 8_762_355

shape: (305_702, 5)
┌──────────────────┬───────────────┬──────────────────┬──────────────────────┬─────────────────────┐
│ Start date       ┆ Start station ┆ End date         ┆ End station number   ┆ Total duration (ms) │
│ ---              ┆ number        ┆ ---              ┆ ---                  ┆ ---                 │
│ str              ┆ ---           ┆ str              ┆ i64                  ┆ i64                 │
│                  ┆ i64           ┆                  ┆                      ┆                     │
╞══════════════════╪═══════════════╪══════════════════╪══════════════════════╪═════════════════════╡
│ 2025-01-14 23:40 ┆ 300050        ┆ 2025-01-15 00:06 ┆ 300050               ┆ 1559817             │
│ 2025-01-14 23:35 ┆ 300059        ┆ 2025-01-15 00:46 ┆ 300059               ┆ 4281047             │
│ 2025-01-14 23:35 ┆ 300059        ┆ 2025-01-15 00:46 ┆ 300059               ┆ 4301835             │
│ 2025-01-14 23:27 ┆ 200166        ┆ 2025-01-14 23:27 ┆ 200166         

We can see that 305702 entries have the same start and end station. This represents 305702/9068057 = 0.033 -> 3.3% of the entries in the dataset (after removing the missing values). We will remove these round trips and we will end up with a dataframe with 8762355 entries in the end.


It is also important to check the consistency of the start and end stations. A good way to check this is making sure that all stations appearing as start station are also appearing as end stations and vice versa. And since we are checking the ids here, we can also check the consistency of these by making sure we have no negative ids.

In [ ]:
########################
#incoherent start and end station numbers (?)
########################

#-------------------------
#Stations appearing in one column but not the other?
#-------------------------

#get uniqueids for both columns
start_ids = df_no_same_station["Start station number"].unique()
end_ids = df_no_same_station["End station number"].unique()

#compare them
start_only = start_ids.filter(~start_ids.is_in(end_ids.implode()))

end_only = end_ids.filter(~end_ids.is_in(start_ids.implode()))


print("Only in Start:", start_only)
print("Only in End:", end_only)

#There are 2 stations that only appear in the end station number column, station ids 10626 and 22168. They are removed a few cells below.

#-------------------------
# Negative station numbers?
#-------------------------

negative_stations = df_no_same_station.filter((pl.col("Start station number") < 0) | (pl.col("End station number") < 0))

print(negative_stations)

# No negative station number ids

Only in Start: shape: (0,)
Series: 'Start station number' [i64]
[
]
Only in End: shape: (2,)
Series: 'End station number' [i64]
[
	10626
	22168
]
shape: (0, 5)
┌────────────┬──────────────────────┬──────────┬────────────────────┬─────────────────────┐
│ Start date ┆ Start station number ┆ End date ┆ End station number ┆ Total duration (ms) │
│ ---        ┆ ---                  ┆ ---      ┆ ---                ┆ ---                 │
│ str        ┆ i64                  ┆ str      ┆ i64                ┆ i64                 │
╞════════════╪══════════════════════╪══════════╪════════════════════╪═════════════════════╡
└────────────┴──────────────────────┴──────────┴────────────────────┴─────────────────────┘


It is seen that there are no negative station identifiers. On the other hand, there are 2 stations that only appear in the end station number column: station ids 10626 and 22168. These 2 stations are not avilable in the file `tfl_bike_stations.csv`, which is the file containing the coordinates of the station and is used to position the stations on a map. Also, since these stations only have the arrival feature, they cannot be used to create the departure-arrival features on notebook 2, so keeping them will bring issues further down the line. A possible explanation of why these 2 stations exists can be that they were used as temporary stations where bikes could be returned but not taken from. Knowing this, is better if we remove them completely, meaning removing **every trip that starts or ends at these stations** (806 trips, < 0.01% of the data)


In [ ]:
# ------------------------------------------------------------
# Remove the stations that only appear in one of the two columns (10626 and 22168)
# ------------------------------------------------------------
stations_to_remove = end_only.to_list() + start_only.to_list()
print("Stations removed:", stations_to_remove)

n_before = df_no_same_station.height

df_no_same_station = df_no_same_station.filter(
    ~pl.col("Start station number").is_in(stations_to_remove)
    & ~pl.col("End station number").is_in(stations_to_remove)
)

print(f"Trips removed: {n_before - df_no_same_station.height}")
print(f"Remaining trips: {df_no_same_station.height:,}")
print(f"Distinct start stations: {df_no_same_station['Start station number'].n_unique()}")
print(f"Distinct end stations:   {df_no_same_station['End station number'].n_unique()}")

Stations removed: [10626, 22168]
Trips removed: 806
Remaining trips: 8,761,549
Distinct start stations: 820
Distinct end stations:   820



Finally, we can also remove trips that are longer than 24 hours and that are most likely a mistake and would skew our results. 


In [ ]:
# ------------------------------------------------
# Remove trips longer than 24 hours
# ------------------------------------------------
MAX_DURATION_MS = 24 * 60 * 60 * 1000  # 24h in milliseconds

long_trips = df_no_same_station.filter(pl.col("Total duration (ms)") > MAX_DURATION_MS)
print(f"Trips longer than 24h: {long_trips.height:,} ({long_trips.height / df_no_same_station.height:.3%} of the trips)")
print("Duration of these trips (hours):")
print((long_trips["Total duration (ms)"] / 3_600_000).describe())

df_no_same_station = df_no_same_station.filter(pl.col("Total duration (ms)") <= MAX_DURATION_MS)
print(f"Remaining trips: {df_no_same_station.height:,}")

Trips longer than 24h: 5,314 (0.061% of the trips)
Duration of these trips (hours):
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 5314.0      │
│ null_count ┆ 0.0         │
│ mean       ┆ 165.640909  │
│ std        ┆ 349.036679  │
│ min        ┆ 24.004739   │
│ 25%        ┆ 37.30734    │
│ 50%        ┆ 68.700648   │
│ 75%        ┆ 166.975293  │
│ max        ┆ 5023.920233 │
└────────────┴─────────────┘
Remaining trips: 8,756,235


We see that there are roughly 5,300 trips (≈0.06% of the data) that last more than 24 hours (the longest one lasts ~209 days). These are not real journeys (bike not docked properly, lost/stolen bike, maintenance) and they heavily distort the *mean* trip duration per station

## Raw data transformation
In this section we will do a first data transformation that will be useful for the coming points. We will first make sure to load and maintain the data in the correct format, specially for the dates. We will then add date and time related features such as date of the trip, day of week (refering to the trip being made on monday, tuesday, etc) and hour of day.

In [12]:
#######################
# Add date columns
#######################
#Transform start and end date from text to date time
df = df_no_same_station.with_columns(
    pl.col("Start date").str.to_datetime("%Y-%m-%d %H:%M"),
    pl.col("End date").str.to_datetime("%Y-%m-%d %H:%M")
)

#Add Operating date, day of week and hour of day columns
df = df.with_columns(
    pl.col("Start date").dt.date().alias("Operating date"),
    pl.col("Start date").dt.weekday().alias("Day of week"),
    pl.col("Start date").dt.hour().alias("Hour of day"),
)

print(df)

shape: (8_756_235, 8)
┌────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────┬───────────┐
│ Start date ┆ Start      ┆ End date   ┆ End        ┆ Total      ┆ Operating  ┆ Day of ┆ Hour of   │
│ ---        ┆ station    ┆ ---        ┆ station    ┆ duration   ┆ date       ┆ week   ┆ day       │
│ datetime[μ ┆ number     ┆ datetime[μ ┆ number     ┆ (ms)       ┆ ---        ┆ ---    ┆ ---       │
│ s]         ┆ ---        ┆ s]         ┆ ---        ┆ ---        ┆ date       ┆ i8     ┆ i8        │
│            ┆ i64        ┆            ┆ i64        ┆ i64        ┆            ┆        ┆           │
╞════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════╪═══════════╡
│ 2025-01-14 ┆ 1043       ┆ 2025-01-15 ┆ 200149     ┆ 870468     ┆ 2025-01-14 ┆ 2      ┆ 23        │
│ 23:59:00   ┆            ┆ 00:13:00   ┆            ┆            ┆            ┆        ┆           │
│ 2025-01-14 ┆ 300015     ┆ 2025-01-15 ┆ 300229     ┆ 307181     ┆ 20

We will also add the stations cordinates in longitude and latitude format so that we can later use it to locate stations in interactive maps.

In [ ]:
#######################
# Add station coordinates
#######################
cord = pl.read_csv("../data/tfl_bike_stations.csv")

print(cord)

#Join start station
df = df.join(cord, left_on='Start station number', right_on = 'station_number', how='left')
df = df.rename({"latitude": "start_station_latitude",
               "longitude": "start_station_longitude",
               "station_name": "start_station_name"})

#Join end station
df = df.join(cord, left_on='End station number', right_on = 'station_number', how='left')
df = df.rename({"latitude": "end_station_latitude",
               "longitude": "end_station_longitude",
               "station_name": "end_station_name"})

df

shape: (799, 4)
┌────────────────┬───────────┬───────────┬─────────────────────────────────┐
│ station_number ┆ latitude  ┆ longitude ┆ station_name                    │
│ ---            ┆ ---       ┆ ---       ┆ ---                             │
│ i64            ┆ f64       ┆ f64       ┆ str                             │
╞════════════════╪═══════════╪═══════════╪═════════════════════════════════╡
│ 1023           ┆ 51.529163 ┆ -0.10997  ┆ River Street , Clerkenwell      │
│ 1018           ┆ 51.499606 ┆ -0.197574 ┆ Phillimore Gardens, Kensington  │
│ 1012           ┆ 51.521283 ┆ -0.084605 ┆ Christopher Street, Liverpool … │
│ 1013           ┆ 51.530059 ┆ -0.120973 ┆ St. Chad's Street, King's Cros… │
│ 3420           ┆ 51.49313  ┆ -0.156876 ┆ Sedding Street, Sloane Square   │
│ …              ┆ …         ┆ …         ┆ …                               │
│ 200145         ┆ 51.537349 ┆ -0.147154 ┆ Gloucester Avenue, Camden Town  │
│ 1188           ┆ 51.536922 ┆ -0.150181 ┆ London Zoo Car Pa

Start date,Start station number,End date,End station number,Total duration (ms),Operating date,Day of week,Hour of day,start_station_latitude,start_station_longitude,start_station_name,end_station_latitude,end_station_longitude,end_station_name
datetime[μs],i64,datetime[μs],i64,i64,date,i8,i8,f64,f64,str,f64,f64,str
2025-01-14 23:59:00,1043,2025-01-15 00:13:00,200149,870468,2025-01-14,2,23,51.517821,-0.096496,"""Museum of London, Barbican""",51.511542,-0.056667,"""Watney Street, Shadwell"""
2025-01-14 23:59:00,300015,2025-01-15 00:04:00,300229,307181,2025-01-14,2,23,51.472509,-0.122831,"""Binfield Road, Stockwell""",51.469202,-0.119022,"""Sidney Road, Stockwell"""
2025-01-14 23:59:00,1068,2025-01-15 00:10:00,1051,640755,2025-01-14,2,23,51.521113,-0.078869,"""Norton Folgate, Liverpool Stre…",51.534042,-0.086379,"""Shoreditch Park, Hoxton"""
2025-01-14 23:59:00,1159,2025-01-15 00:10:00,1007,642548,2025-01-14,2,23,51.522853,-0.099994,"""Berry Street, Clerkenwell""",51.51477,-0.122219,"""Drury Lane, Covent Garden"""
2025-01-14 23:58:00,200048,2025-01-15 00:14:00,1058,944185,2025-01-14,2,23,51.493978,-0.127554,"""Page Street, Westminster""",51.510212,0.004979,"""Orchard Place, Blackwall Tunne…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-12-16 00:02:00,200141,2025-12-16 00:16:00,200237,837578,2025-12-16,2,0,51.509158,-0.224103,"""Westfield Ariel Way, White Cit…",51.472817,-0.199783,"""Parson's Green , Parson's Gree…"
2025-12-16 00:01:00,200011,2025-12-16 00:16:00,200250,908383,2025-12-16,2,0,51.519265,-0.021345,"""Furze Green, Bow""",51.520893,-0.051394,"""Cleveland Way, Stepney"""
2025-12-16 00:00:00,300096,2025-12-16 00:07:00,962,432164,2025-12-16,2,0,51.511891,-0.107349,"""Tallis Street, Temple""",51.505569,-0.111606,"""Stamford Street, South Bank"""


Not every station id exists in `tfl_bike_stations.csv` file used for the coordinates. We can check the stations without coordinates: they stay in the analysis (their trips are valid) but they **cannot be drawn on the maps** of notebook 2.

In [ ]:
# stations without coordinates / name
all_station_ids = pl.concat([df["Start station number"], df["End station number"]]).unique()
missing_coords = all_station_ids.filter(~all_station_ids.is_in(cord["station_number"].implode())).sort()

trips_without_coords = df.filter(
    pl.col("start_station_latitude").is_null() | pl.col("end_station_latitude").is_null()
).height

print(f"Stations in the trip data: {all_station_ids.len()}")
print(f"Stations without coordinates: {missing_coords.len()}")
print(missing_coords.to_list())
print(f"Trips touching one of these stations: {trips_without_coords:,} ({trips_without_coords / df.height:.2%})")

Stations in the trip data: 820
Stations without coordinates: 23
[1088, 1152, 1158, 3478, 22163, 22181, 200015, 300232, 1059444, 1085444, 1190444, 1219444, 11274444, 200025444, 200083444, 200134444, 200136444, 200141444, 200191444, 200230444, 300032444, 300241444, 2001454444]
Trips touching one of these stations: 234,917 (2.68%)


We can see that many of them have ids ending in `444` (e.g. 200083444, 11274444), this could mean that these stations are temporary / relocated versions of existing stations and that could explain why they are not available in the coordinates file used.

In [15]:
#######################
#export df
#######################

#df.write_csv("../data/output/v2/2.clean_df_v2.csv")
df.write_parquet("../data/output/v2/2.clean_df_v2.parquet")
